# 04 · Why does a perturbation hurt? Probing the policy (Step 2d)

<a href="https://colab.research.google.com/github/danielamrh/act-visual-robustness/blob/main/notebooks/04_probe.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>

Every perturbation is purely visual, so a clean episode's actions can be **replayed** under any perturbation and the
robot passes through exactly the same states - only the camera image differs. At 8 probe steps per episode we feed
the clean and the perturbed image (same state) through the policy and measure:

| Metric | Meaning |
|---|---|
| `feature_shift` | cosine distance between the encoder's feature maps - how much the *representation* moves |
| `action_shift` | mean abs. difference of the predicted 100-step action chunks (rad) - how much the *decision* moves |
| `attn_shift` | total variation between the decoder's cross-attention maps |

Plus, for the first seed, **occlusion sensitivity** maps (how much the action chunk changes when a gray patch hides each
image location), with the shadows outlined. No closed-loop rollouts under perturbation are needed: ~10 min per policy.

## 0 · Environment check

In [ ]:
import sys, subprocess
print(sys.version)
assert sys.version_info >= (3, 12), "LeRobot 0.6.x needs Python >= 3.12"
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout or "No GPU - switch the runtime to T4!")

## 1 · Drive + repo + install
Code comes from GitHub (always fresh via `git pull`), outputs go to Google Drive.

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")

REPO = "act-visual-robustness"
REPO_DIR = f"/content/{REPO}"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/danielamrh/{REPO}.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
COMMIT = !git rev-parse --short HEAD
COMMIT = COMMIT[0]
print("commit:", COMMIT)

# labmaze (pulled in by dm-control) has no wheel for Python >= 3.13 and needs Bazel
# to build. gym-aloha never uses it, so install an empty placeholder first.
if sys.version_info >= (3, 13):
    !pip install -q ./tools/labmaze_stub

# installs lerobot[aloha] pinned in pyproject.toml (takes a few minutes) and the act_enc policy plugin
!pip install -q -e ".[sim]" -e plugins/lerobot_policy_act_enc
sys.path.insert(0, f"{REPO_DIR}/src")  # editable install is only picked up after a restart
# re-running this cell in a live session must pick up the freshly pulled code:
# drop already imported avr modules so the next import loads the new version
for name in [m for m in sys.modules if m == "avr" or m.startswith("avr.")]:
    del sys.modules[name]

# GPU rendering: without NVIDIA's EGL registration MuJoCo silently renders on the CPU
from avr.colab import ensure_nvidia_egl, gl_renderer
ensure_nvidia_egl()
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"
renderer = gl_renderer()
print("OpenGL renderer:", renderer)
if "llvmpipe" in renderer.lower() or renderer.startswith("failed"):
    print("⚠ MuJoCo is NOT rendering on the GPU - rollouts will be extremely slow")

DRIVE_ROOT = "/content/drive/MyDrive/act_robustness"
os.makedirs(DRIVE_ROOT, exist_ok=True)

### Quick check: can we create and render the ALOHA env?

In [ ]:
import gymnasium as gym
import gym_aloha  # registers gym_aloha/* envs
import matplotlib.pyplot as plt

import time
env = gym.make("gym_aloha/AlohaTransferCube-v0")
obs, _ = env.reset(seed=0)
img = obs["top"] if isinstance(obs, dict) and "top" in obs else env.render()

# speed check: env step incl. observation rendering, zero actions
t0 = time.perf_counter()
for _ in range(50):
    env.step(env.action_space.sample() * 0)
dt = (time.perf_counter() - t0) / 50
env.close()
print(f"{dt*1000:.0f} ms per env step -> one 400-step episode ≈ {400*dt:.0f} s (+ video rendering)")
plt.imshow(img); plt.axis("off"); plt.title("AlohaTransferCube-v0, seed 0");

## 2 · Run the probe for every trained policy (resumable per seed)

In [ ]:
from avr.background import start_background, watch
from avr.registry import policy_registry, trained

POLICIES = trained(policy_registry(DRIVE_ROOT))
print(*POLICIES, sep="\n")
N_SEEDS = 10

for label, paths in POLICIES.items():
    print(f"\n=== {label}")
    cmd = [sys.executable, "-m", "avr.analysis.features", "--policy", paths.checkpoint,
           "--out", paths.probe, "--seeds", str(N_SEEDS), "--seed", "1000"]
    log = f"/content/logs/probe_{os.path.basename(paths.probe)}.log"
    watch(start_background(cmd, log), log, interval=60)

## 3 · Probe metrics per perturbation
Mean over 10 seeds (each seed averaged over its probe steps first), per policy.

In [ ]:
import pandas as pd
from IPython.display import display
from avr.analysis.attention import load_probe_rows, shift_table

probe_rows = {label: load_probe_rows(f"{p.probe}/feature_probe.jsonl") for label, p in POLICIES.items()
              if os.path.exists(f"{p.probe}/feature_probe.jsonl")}
for label, rows in probe_rows.items():
    print(label)
    df = pd.DataFrame(shift_table(rows))[["factor", "level", "feature_shift", "action_shift", "attn_shift"]]
    display(df.style.format({"feature_shift": "{:.3f}", "action_shift": "{:.4f}", "attn_shift": "{:.3f}"}).hide(axis="index"))

## 4 · Does the probe predict the damage?
One point per factor x level: probe shift (x) against the paired success change from the perturbation suite
(notebook 03, y). A high rank correlation means the cheap probe explains where the policy fails.
Only policies with suite results are shown.

In [ ]:
from avr.analysis.attention import plot_shift_vs_drop
from avr.analysis.robustness import load_cells, summary_rows

pairs = {}
for label, rows in probe_rows.items():
    cells = load_cells(POLICIES[label].robustness)
    if ("clean", 0) not in cells:
        continue
    suite = summary_rows({k: v for k, v in cells.items() if k[0] != "clean"}, clean=cells[("clean", 0)])
    pairs[label] = (rows, suite)

PROBE_FIG = f"{DRIVE_ROOT}/probe"
for metric in ("action_shift", "feature_shift"):
    fig, rhos = plot_shift_vs_drop(pairs, metric=metric, path=f"{PROBE_FIG}/{metric}_vs_drop.png")
    print(metric, {k: round(v, 2) for k, v in rhos.items()})

## 5 · Where does the policy look?
Occlusion sensitivity (bottom) for the clean image and five perturbations, first seed, probe steps 50 and 100.
Orange outlines: pixels that change when shadows are switched off, i.e. the shadows.

In [ ]:
import glob
from avr.analysis.attention import attention_figure, load_figure_maps

for label, paths in POLICIES.items():
    figs = sorted(glob.glob(f"{paths.probe}/figures_seed*.npz"))
    if not figs:
        continue
    maps = load_figure_maps(figs[0])
    for step in sorted(maps["None_L0"]):
        attention_figure(maps, step, kind="occl", title=f"{label}: occlusion sensitivity, step {step}",
                         path=f"{paths.probe}/occlusion_step{step}.png")

### 5b · For comparison: the decoder cross-attention (diffuse, piles up on empty background)

In [ ]:
for label, paths in POLICIES.items():
    figs = sorted(glob.glob(f"{paths.probe}/figures_seed*.npz"))
    if figs:
        maps = load_figure_maps(figs[0])
        attention_figure(maps, sorted(maps["None_L0"])[0], kind="attn", title=f"{label}: cross-attention")